In [4]:
# Why: Load the prepared text chunks so they can be converted into embeddings.
import json
from pathlib import Path

chunks_path = Path.cwd().parent / "storage" / "processed" / "chunks.json"
with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [17]:
# Why: Confirm which Python environment runs the notebook before installing or importing packages.
import sys
print(sys.executable)


/workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/llm-zoomcamp-2026-code/.venv/bin/python


In [27]:
# Why: Install the library that creates semantic vector embeddings from text.
import sys

!{sys.executable} -m pip install sentence-transformers

  Obtaining dependency information for sentence-transformers from https://files.pythonhosted.org/packages/c1/ad/8f73f512dc7ad4031d2b64cbb67f70bdfb355756afbe0db610a5146415c1/sentence_transformers-5.6.1-py3-none-any.whl.metadata
  Obtaining dependency information for transformers<6.0.0,>=4.41.0 from https://files.pythonhosted.org/packages/6f/67/8d85ca2323233ae3c0365a659c4e52ee1f587b440e4bc577e7d8e4416d0f/transformers-5.14.1-py3-none-any.whl.metadata
  Obtaining dependency information for torch>=1.11.0 from https://files.pythonhosted.org/packages/f3/82/fea946351658e6534db52d2cc12bc53087cbf87f9440c5f180f367c1950b/torch-2.13.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for scikit-learn>=0.22.0 from https://files.pythonhosted.org/packages/6c/c2/63fdda36c56437eeb44aaf9493c8bcd62ce230ab1598924fc626ffbfa943/scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for scipy>=1.0.0 from https:

In [9]:
# Why: Locate the project and import its shared embedding model instead of redefining it here.
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "storage").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ingestion.embeddings import model
print(model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [10]:
# Why: Create one sample embedding to verify that the model works and see its vector size.
embedding = model.encode("Hello")

print(len(embedding))

384


In [14]:
# Why: Test the helper with a small batch before processing every document chunk.
from ingestion.embeddings import embed_documents


texts = [
    "Hello world",
    "Large Language Models"
]

vectors = embed_documents(texts)
print(len(vectors))

2


In [16]:
# Why: Reload the source chunks that will receive embeddings in the final dataset.
import json

with open("../storage/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(len(chunks))   

88


In [21]:
# Why: Extract chunk text and encode it so semantic search can compare chunks by meaning.
texts = [chunk["text"] for chunk in chunks]
vectors = embed_documents(texts)

In [18]:
# Why: Check that every chunk has a vector and that the vectors have the expected dimensions.
print(len(vectors))
print(len(vectors[0]))

88
384


In [19]:
# Why: Attach each vector to its matching chunk so text and embedding stay together.
for chunk, vector in zip(chunks, vectors):
    chunk["embedding"] = vector

In [22]:
# Why: Save the enriched chunks so later retrieval steps can load them without re-embedding.
import json

with open(
    "../storage/processed/chunks_with_embeddings.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        chunks,
        f,
        indent=2,
        ensure_ascii=False
    )